In [ ]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split

PM10 = pd.read_csv('Cont.temuco/PM10.csv', sep=';')
PM10.head(5)

,FECHA (YYMMDD),HORA (HHMM),Registros validados,Registros preliminares,Registros no validados,Unnamed: 5
0,10101,0,NaN,"22,9729",NaN,NaN
1,10102,0,NaN,"23,3625",NaN,NaN
2,10103,0,NaN,"63,2228",NaN,NaN
3,10104,0,NaN,"50,2354",NaN,NaN
4,10105,0,NaN,"24,7458",NaN,NaN


In [19]:
PM25 = pd.read_csv('Cont.temuco/PM2.5.csv', sep=';')
PM25.head(5)

,FECHA (YYMMDD),HORA (HHMM),Registros validados,Registros preliminares,Registros no validados,Unnamed: 5
0,90101,0,NaN,"14,0333",NaN,NaN
1,90102,0,NaN,"12,4833",NaN,NaN
2,90103,0,NaN,"12,0042",NaN,NaN
3,90104,0,NaN,"9,23332",NaN,NaN
4,90105,0,NaN,"10,4",NaN,NaN


In [21]:
import os

folder = 'Cont.temuco'
csv_files = [f for f in os.listdir(folder) if f.endswith('.csv') and f not in ['PM2.5.csv', 'PM10.csv']]

dataframes = {}
for file in csv_files:
    df_name = os.path.splitext(file)[0]
    dataframes[df_name] = pd.read_csv(os.path.join(folder, file), sep=';')

    print(f"DataFrame: {df_name}")
    print(dataframes[df_name].head(), "\n")

DataFrame: CO
   FECHA (YYMMDD)  HORA (HHMM) Registros validados Registros preliminares  \
0           40430            0                 NaN                3,69583   
1           40501            0                 NaN                1,07083   
2           40502            0                 NaN                1,09167   
3           40503            0                 NaN                 2,7625   
4           40504            0                 NaN               0,866666   

  Registros no validados  Unnamed: 5  
0                    NaN         NaN  
1                    NaN         NaN  
2                    NaN         NaN  
3                    NaN         NaN  
4                    NaN         NaN   

DataFrame: NO
   FECHA (YYMMDD)  HORA (HHMM)  Registros validados Registros preliminares  \
0           40401            0                  NaN                   1,05   
1           40402            0                  NaN              0,0249999   
2           40403            0         

In [31]:
import pandas as pd
import glob
from functools import reduce
from datetime import datetime

# Configuración inicial
ruta_archivos = './'  # Cambia esto a tu ruta real
archivos_contaminantes = ['CO.csv', 'NO.csv', 'NO2.csv', 'NOX.csv', 'PM2.5.csv', 'PM10.csv']

# Función para leer y procesar cada archivo
def procesar_contaminante(archivo):
    try:
        # Extraer el nombre del contaminante del nombre del archivo
        contaminante = archivo.replace('.csv', '')
        
        # Leer el archivo CSV
        df = pd.read_csv(archivo, sep=';', decimal=',')
        
        # Verificar columnas requeridas
        if not all(col in df.columns for col in ['FECHA (YYMMDD)', 'HORA (HHMM)', 'Registros validados']):
            print(f"Advertencia: {archivo} no tiene las columnas requeridas")
            return None
        
        # Seleccionar y renombrar columnas
        df = df[['FECHA (YYMMDD)', 'HORA (HHMM)', 'Registros validados']].copy()
        df = df.rename(columns={'Registros validados': contaminante})
        
        # Limpiar y convertir valores numéricos
        df[contaminante] = (
            df[contaminante].astype(str)
            .str.replace(',', '.')
            .replace(['nan', 'NaN', 'NA', '', ' '], pd.NA)
        )
        df[contaminante] = pd.to_numeric(df[contaminante], errors='coerce')
        
        # Crear columna de fecha legible
        df['FECHA_LEGIBLE'] = pd.to_datetime(
            df['FECHA (YYMMDD)'].astype(str).str.zfill(6),
            format='%y%m%d',
            errors='coerce'
        )
        
        # Eliminar filas con fechas inválidas
        df = df.dropna(subset=['FECHA_LEGIBLE'])
        
        # Eliminar columnas originales (ya no son necesarias)
        df = df.drop(columns=['FECHA (YYMMDD)', 'HORA (HHMM)'])
        
        return df
    
    except Exception as e:
        print(f"Error procesando {archivo}: {str(e)}")
        return None

# Procesar todos los archivos
dataframes = []
for archivo in archivos_contaminantes:
    df_procesado = procesar_contaminante(archivo)
    if df_procesado is not None:
        dataframes.append(df_procesado)
        print(f"Procesado: {archivo} ({len(df_procesado)} registros válidos)")

# Unir todos los DataFrames por fecha
if dataframes:
    # Usar reduce para hacer merge sucesivo
    df_final = reduce(
        lambda left, right: pd.merge(left, right, on='FECHA_LEGIBLE', how='outer'), 
        dataframes
    ).sort_values('FECHA_LEGIBLE')
    
    # Mostrar resultados
    print("\nDataFrame combinado:")
    print(df_final.info())
    
    print("\nPrimeras filas:")
    print(df_final.head())
    
    print("\nResumen estadístico:")
    print(df_final.describe())
    
    # Opcional: Guardar el resultado
    df_final.to_csv('contaminantes_combinados.csv', index=False)
    print("\nDatos guardados en 'contaminantes_combinados.csv'")

    # Visualización básica (opcional)
    try:
        import matplotlib.pyplot as plt

        df_final.set_index('FECHA_LEGIBLE').plot(subplots=True, figsize=(12, 8))
        plt.suptitle('Niveles de Contaminantes')
        plt.tight_layout()
        plt.show()
    except ImportError:
        print("Para visualización, instala matplotlib: pip install matplotlib")
else:
    print("No se pudo procesar ningún archivo")

Error procesando CO.csv: [Errno 2] No such file or directory: 'CO.csv'
Error procesando NO.csv: [Errno 2] No such file or directory: 'NO.csv'
Error procesando NO2.csv: [Errno 2] No such file or directory: 'NO2.csv'
Error procesando NOX.csv: [Errno 2] No such file or directory: 'NOX.csv'
Error procesando PM2.5.csv: [Errno 2] No such file or directory: 'PM2.5.csv'
Error procesando PM10.csv: [Errno 2] No such file or directory: 'PM10.csv'
No se pudo procesar ningún archivo


In [33]:
df_final

NameError: name 'df_final' is not defined